# Assignment 3 - 02477 Bayesian Machine Learning

In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np
from jax import random
import seaborn as snb

In [4]:
# Load data
data = jnp.load('./data_assignment3.npz')
x, y = data['x'], data['t']
print(x.shape)
print(y.shape)

(200,)
(200,)


In [5]:
xt = jnp.vstack((x, jnp.ones(200,))).T
print(xt.shape)

(200, 2)


> Task 1.1: Marginalize out each $z_n$ from the joint model in equation (7)

$$
K = \mathcal{N}(v|0,\tau^2I)\mathcal{N}(w_0|0,\tau^2I)\mathcal{N}(w_1|0,\tau^2I)\mathcal{N}_{+}(\sigma_0|0,1)\mathcal{N}_{+}(\sigma_1|0,1)\mathcal{N}_{+}(\tau|0,1)
$$

$$
p(y,z,v,w_0,w_1,\sigma_0,\sigma_1) = \left[\prod_{n=1}^{N}\mathcal{N}(y_n|w_{z_n}^Tx_n, \sigma_{z_n}^2)Ber(z_n| \sigma(v^Tx_n))\right]\cdot K
$$

$$
p(y,v,w_0,w_1,\sigma_0,\sigma_1) = \sum_{z_1\in{0,1}}\sum_{z_2\in{0,1}}...\sum_{z_N\in{0,1}}p(y,z,v,w_0,w_1,\sigma_0,\sigma_1)
$$

$$
p(y,v,w_0,w_1,\sigma_0,\sigma_1) = \sum_{z_1\in{0,1}}\sum_{z_2\in{0,1}}...\sum_{z_N\in{0,1}}\left[\prod_{n=1}^{N}\mathcal{N}(y_n|w_{z_n}^T x_n, \sigma_{z_n}^2)Ber(z_n| \sigma(v^Tx_n))\right]\cdot K
$$

$$
p(y,v,w_0,w_1,\sigma_0,\sigma_1) = K \cdot \sum_{z_1\in{0,1}}\sum_{z_2\in{0,1}}...\sum_{z_N\in{0,1}}\left[\prod_{n=1}^{N}\mathcal{N}(y_n|w_{z_n}^T x_n, \sigma_{z_n}^2)Ber(z_n| \sigma(v^Tx_n))\right]
$$


$$
p(y,v,w_0,w_1,\sigma_0,\sigma_1) = K\cdot \left[\prod_{n=1}^{N}\sum_{z_n\in{0,1}}\mathcal{N}(y_n|w_{z_n}^T x_n, \sigma_{z_n}^2)Ber(z_n| \sigma(v^Tx_n))\right]
$$

For each value of n:
$$
\sum_{z_n\in{0,1}}\mathcal{N}(y_n|w_{z_n}^T x_n, \sigma_{z_n}^2)Ber(z_n| \sigma(v^Tx_n)) = (1-\sigma(v^Tx_n))\mathcal{N}(y_n|w_0^T x_n, \sigma_0^2) + \sigma(v^Tx_n)\mathcal{N}(y_n|w_1^T x_n, \sigma_1^2)
$$

Then we arrive at:
$$
p(y,v,w_0,w_1,\sigma_0,\sigma_1) = \left[\prod_{n=1}^{N}(1-\sigma(v^Tx_n))\mathcal{N}(y_n|w_0^T x_n, \sigma_0^2) + \sigma(v^Tx_n)\mathcal{N}(y_n|w_1^T x_n, \sigma_1^2)\right]\cdot K
$$



> Task 1.2: Implement a Python function to evaluate the marginalized log joint distribution

In [28]:
log_npdf = lambda x, m, v: -(x-m)**2/(2*v) - 0.5*jnp.log(2*jnp.pi*v)
npdf = lambda x, m, v: jnp.exp(log_npdf(x, m, v))
log_half_npdf = lambda x, m, v: jnp.log(2) -0.5*(x-m)**2/(v) - 0.5*jnp.log(2*jnp.pi*v)
sigmoid = lambda x: 1./(1 + jnp.exp(-x))


def log_joint(theta):

    w0, w1, v, tau, sig02, sig12 = theta

    sig0 = jnp.sqrt(sig02)
    sig1 = jnp.sqrt(sig12)
    # w0, w1, v are vectors
    
    sigs = sigmoid(v.T@xt)

    # We need to sample yn
    nw0 = jnp.sum(log_npdf(w0, 0, tau**2))
    nw1 = jnp.sum(log_npdf(w1, 0, tau**2))
    nv = jnp.sum(log_npdf(v, 0, tau**2))

    ntau = log_half_npdf(tau, 0, 1)
    nsig0 = log_half_npdf(sig0, 0, 1)
    nsig1 = log_half_npdf(sig1, 0, 1)

    likelihood = jnp.sum(jnp.log((1-sigs)*npdf(y, w0.T@xt, sig02) + sigs*npdf(y,w1.T@xt,sig12)))

    return likelihood + nw0 + nw1 + nv + ntau + nsig0 + nsig1
